<a href="https://colab.research.google.com/github/melatisurya/Data_Metopen_Klasifikasi_Penyakit_Daun_Jagung/blob/main/Maize_ResNet50_Fixed_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Klasifikasi Penyakit Daun Jagung dengan ResNet50 (versi perbaikan)

Notebook ini menjalankan pipeline yang sudah diperbaiki sesuai catatan reviewer:

1. **Praproses benar**: `resnet50.preprocess_input` (bukan pembagian 255).
2. **Validasi silang sah**: model diinisialisasi ulang pada tiap fold; augmentasi hanya dibuat dari citra latih fold tersebut.
3. **Fine-tuning** blok terakhir (`conv5_block3`) setelah kepala klasifikasi dilatih.
4. Data uji (64 citra) tidak dipakai untuk memilih model; model terbaik dipilih dari akurasi validasi.
5. Metrik: akurasi (benar/n), presisi/recall/F1 berbobot, AUC OvR makro, interval Wilson 95%.

**Cara pakai:** Runtime > Change runtime type (GPU opsional, CPU juga bisa) lalu jalankan sel dari atas ke bawah. Unggah zip dataset (folder berisi `Hawar`, `Karat`, `Sehat`) saat diminta. Hasil (CSV, gambar, ringkasan) ada di folder `hasil_resnet50` dan diunduh di sel terakhir.

> Catatan efisiensi: lapisan sebelum `conv5_block3` selalu beku, sehingga keluarannya di-cache sekali. Melatih `conv5_block3` + kepala di atas cache ini setara matematis dengan melatih model utuh dengan lapisan awal beku (BatchNorm dibekukan), tetapi jauh lebih cepat.

## 1. Instalasi

In [ ]:
!pip -q install split-folders opencv-python-headless

## 2. Unggah dan ekstrak dataset

Pilih berkas zip dataset (misalnya `archive (2).zip`). Jika nama folder di dalam zip berbeda, ubah `DATA_DIR` di sel berikutnya.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()   # pilih zip dataset
for name in up:
    zipfile.ZipFile(name).extractall('data')
print(os.listdir('data'))

Saving archive.zip to archive (1).zip
['Penyakit Daun Jagung New']


## 3. Pengaturan

`K_AUG` = jumlah salinan augmentasi per citra latih, `N_FOLDS` = jumlah fold, `EPOCHS_FT` = epoch maksimum fine-tuning.

In [ ]:
import os
os.environ['DATA_DIR'] = 'data/Penyakit Daun Jagung New'   # sesuaikan jika perlu
os.environ['OUT_DIR']  = 'hasil_resnet50'
os.environ['WEIGHTS']  = 'imagenet'                        # bobot ImageNet resmi Keras
os.environ['K_AUG']    = '3'
os.environ['N_FOLDS']  = '5'
os.environ['EPOCHS_FT'] = '8'

## 4. Impor dan konstanta

In [ ]:
import os, time, json, math
import numpy as np, pandas as pd, cv2
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, roc_auc_score,
                             confusion_matrix)
from sklearn.linear_model import LogisticRegression
import splitfolders
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA_DIR = os.environ.get("DATA_DIR", "data/Penyakit Daun Jagung New")
OUT_DIR  = os.environ.get("OUT_DIR", "hasil_resnet50")
WEIGHTS  = os.environ.get("WEIGHTS", "imagenet")
K_AUG    = int(os.environ.get("K_AUG", 3))         # jumlah salinan augmentasi per citra latih
N_FOLDS  = int(os.environ.get("N_FOLDS", 5))
SEED     = 69
IMG      = 224
EPOCHS_HEAD, PAT_HEAD, LR_HEAD = 75, 10, 1e-4
EPOCHS_FT,   PAT_FT,   LR_FT   = int(os.environ.get("EPOCHS_FT", 8)), 3, 5e-5
BATCH    = 16
os.makedirs(OUT_DIR, exist_ok=True)
T0 = time.time()
def log(*a): print(f"[{time.time()-T0:6.0f}s]", *a, flush=True)
keras.utils.set_random_seed(SEED)


## 5. Data dan pembagian 80:20 (seed 69)

In [ ]:
# ------------------------------------------------------------------ 1. data dan pembagian 80:20 (seed 69)
split_dir = os.path.join(OUT_DIR, "split")
if not os.path.exists(split_dir):
    splitfolders.ratio(DATA_DIR, output=split_dir, seed=SEED, ratio=(.8, .2), move=False)
classes = sorted(d for d in os.listdir(os.path.join(split_dir, "train")))
CN = {"Hawar": "Leaf blight", "Karat": "Leaf rust", "Sehat": "Healthy"}
def load_split(name):
    X, y = [], []
    for i, c in enumerate(classes):
        for f in sorted(os.listdir(os.path.join(split_dir, name, c))):
            im = cv2.imread(os.path.join(split_dir, name, c, f))
            im = cv2.resize(im, (IMG, IMG), interpolation=cv2.INTER_AREA)   # area interpolation
            X.append(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); y.append(i)
    return np.stack(X), np.array(y)
Xtr, ytr = load_split("train"); Xte, yte = load_split("val")
log("kelas", classes, "| latih", Xtr.shape[0], np.bincount(ytr), "| uji", Xte.shape[0], np.bincount(yte))


[    43s] kelas ['Hawar', 'Karat', 'Sehat'] | latih 246 [85 81 80] | uji 64 [22 21 21]


## 6. Augmentasi (parameter mengikuti Augmentor)

In [ ]:
# ------------------------------------------------------------------ 2. augmentasi (parameter mengikuti Augmentor)
def augment(img, rng):
    h, w = img.shape[:2]
    # skew 0,15
    m = 0.15 * rng.uniform(0.5, 1.0)
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    d = rng.integers(0, 8); s = m * w
    dst = src.copy()
    if d == 0: dst[0, 0] += s; dst[3, 0] += s            # miring ke kanan (sisi kiri)
    elif d == 1: dst[1, 0] -= s; dst[2, 0] -= s
    elif d == 2: dst[0, 1] += s; dst[1, 1] += s
    elif d == 3: dst[2, 1] -= s; dst[3, 1] -= s
    elif d == 4: dst[0] += [s, s]
    elif d == 5: dst[1] += [-s, s]
    elif d == 6: dst[2] += [-s, -s]
    else: dst[3] += [s, -s]
    out = cv2.warpPerspective(img, cv2.getPerspectiveTransform(src, dst), (w, h), borderMode=cv2.BORDER_REFLECT)
    # kecerahan 0,75-1,25
    out = np.clip(out.astype(np.float32) * rng.uniform(0.75, 1.25), 0, 255).astype(np.uint8)
    # rotasi +-3 derajat dan zoom 0,95-1,05
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-3, 3), rng.uniform(0.95, 1.05))
    out = cv2.warpAffine(out, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    if rng.random() < 0.5: out = out[:, ::-1]
    return np.ascontiguousarray(out)
rng = np.random.default_rng(SEED)
Xaug = np.stack([augment(Xtr[i], rng) for i in range(len(Xtr)) for _ in range(K_AUG)])
src_idx = np.repeat(np.arange(len(Xtr)), K_AUG)               # sumber tiap salinan augmentasi
log("salinan augmentasi:", Xaug.shape[0])


[    45s] salinan augmentasi: 738


## 7. ResNet50 dan cache fitur

Bagian ini paling lama (ekstraksi fitur citra asli, salinan augmentasi, dan uji praproses lama `/255`).

In [ ]:
# ------------------------------------------------------------------ 3. ResNet50 dan cache fitur
base = keras.applications.ResNet50(include_top=False, weights=WEIGHTS, input_shape=(IMG, IMG, 3))
f2 = keras.Model(base.input, base.get_layer("conv5_block2_out").output)
b3 = keras.Model(base.get_layer("conv5_block2_out").output, base.output)    # conv5_block3 saja
b3_init = b3.get_weights()
def feats(X, mode):
    X = X.astype("float32")
    X = keras.applications.resnet50.preprocess_input(X) if mode == "correct" else X / 255.0
    return f2.predict(X, batch_size=16, verbose=0)
cache = os.path.join(OUT_DIR, "cache.npz")
if os.path.exists(cache):
    z = np.load(cache); F2tr, F2aug, F2te, Pbug_tr, Pbug_aug, Pbug_te = [z[k] for k in
        ("F2tr", "F2aug", "F2te", "Pbug_tr", "Pbug_aug", "Pbug_te")]
else:
    F2tr = feats(Xtr, "correct").astype("float16"); log("fitur latih selesai")
    F2aug = feats(Xaug, "correct").astype("float16"); log("fitur augmentasi selesai")
    F2te = feats(Xte, "correct").astype("float16"); log("fitur uji selesai")
    pool = lambda F2: b3.predict(F2, batch_size=32, verbose=0).mean((1, 2))
    Pbug_tr = pool(feats(Xtr, "bug")); Pbug_aug = pool(feats(Xaug, "bug")); Pbug_te = pool(feats(Xte, "bug"))
    log("fitur /255 (uji kesalahan praproses lama) selesai")
    np.savez(cache, F2tr=F2tr, F2aug=F2aug, F2te=F2te, Pbug_tr=Pbug_tr, Pbug_aug=Pbug_aug, Pbug_te=Pbug_te)
pool = lambda F2: b3.predict(F2.astype("float32"), batch_size=32, verbose=0).mean((1, 2))
Ptr, Paug, Pte = pool(F2tr), pool(F2aug), pool(F2te)           # fitur beku (GAP conv5_block3_out)


## 8. Validasi silang 5-fold: kepala (F) lalu fine-tuning conv5_block3 (T)

Konfigurasi: `F-aug-/255` (praproses lama), `F-noaug`, `F-aug`, `T-noaug`, `T-aug`.

In [ ]:
# ------------------------------------------------------------------ 4. model kepala dan fine-tuning
def head():
    return keras.Sequential([keras.layers.Input((2048,)), keras.layers.Dropout(0.5),
                             keras.layers.Dense(512, activation="relu"), keras.layers.Dropout(0.5),
                             keras.layers.Dense(3, activation="softmax")])
def stop(p): return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=p, restore_best_weights=True)]
def metrics(y, proba):
    pred = proba.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="weighted", zero_division=0)
    return dict(acc=accuracy_score(y, pred), prec_w=p, rec_w=r, f1_w=f,
                auc=roc_auc_score(y, proba, multi_class="ovr", average="macro"))
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100 * (c - h), 100 * (c + h)

rows, store = [], {}
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (itr, iva) in enumerate(skf.split(Xtr, ytr), 1):
    for aug in (False, True):
        keras.utils.set_random_seed(SEED + fold)
        # data latih fold: citra asli fold + salinan augmentasi HANYA dari citra asli fold ini
        if aug:
            sel = np.isin(src_idx, itr)
            Xp = np.concatenate([Ptr[itr], Paug[sel]]); Xb = np.concatenate([Pbug_tr[itr], Pbug_aug[sel]])
            Xf = np.concatenate([F2tr[itr], F2aug[sel]]); yy = np.concatenate([ytr[itr], ytr[src_idx[sel]]])
        else:
            Xp, Xb, Xf, yy = Ptr[itr], Pbug_tr[itr], F2tr[itr], ytr[itr]
        tag = "aug" if aug else "noaug"
        # ---- F: kepala saja (backbone beku), kepala diinisialisasi ulang tiap fold
        h = head(); h.compile(keras.optimizers.Adam(LR_HEAD), "sparse_categorical_crossentropy", metrics=["accuracy"])
        hist = h.fit(Xp, yy, validation_data=(Ptr[iva], ytr[iva]), epochs=EPOCHS_HEAD, batch_size=BATCH,
                     callbacks=stop(PAT_HEAD), verbose=0)
        pv, pt = h.predict(Ptr[iva], verbose=0), h.predict(Pte, verbose=0)
        cfg = f"F-{tag}"
        rows.append(dict(config=cfg, fold=fold, val_acc=accuracy_score(ytr[iva], pv.argmax(1)),
                         val_loss=min(hist.history["val_loss"]), epochs=len(hist.history["loss"]),
                         **{"test_" + k: v for k, v in metrics(yte, pt).items()}))
        store[(cfg, fold)] = dict(proba=pt, hist=hist.history)
        hw = h.get_weights()
        # ---- uji kesalahan praproses lama (/255) pada desain F lama: hanya skenario augmentasi
        if aug:
            hb = head(); hb.compile(keras.optimizers.Adam(LR_HEAD), "sparse_categorical_crossentropy", metrics=["accuracy"])
            hh = hb.fit(Xb, yy, validation_data=(Pbug_tr[iva], ytr[iva]), epochs=EPOCHS_HEAD, batch_size=BATCH,
                        callbacks=stop(PAT_HEAD), verbose=0)
            pv2, pt2 = hb.predict(Pbug_tr[iva], verbose=0), hb.predict(Pbug_te, verbose=0)
            rows.append(dict(config="F-aug-/255", fold=fold, val_acc=accuracy_score(ytr[iva], pv2.argmax(1)),
                             val_loss=min(hh.history["val_loss"]), epochs=len(hh.history["loss"]),
                             **{"test_" + k: v for k, v in metrics(yte, pt2).items()}))
        # ---- T: fine-tuning conv5_block3 (BatchNorm beku) + kepala, mulai dari kepala F
        inp = keras.Input((7, 7, 2048))
        blk = keras.models.clone_model(b3); blk.set_weights(b3_init)
        for l in blk.layers:
            if isinstance(l, keras.layers.BatchNormalization): l.trainable = False
        hd = head(); hd.set_weights(hw)
        out = hd(keras.layers.GlobalAveragePooling2D()(blk(inp)))
        ft = keras.Model(inp, out)
        ft.compile(keras.optimizers.Adam(LR_FT), "sparse_categorical_crossentropy", metrics=["accuracy"])
        hf = ft.fit(Xf.astype("float32"), yy, validation_data=(F2tr[iva].astype("float32"), ytr[iva]),
                    epochs=EPOCHS_FT, batch_size=BATCH, callbacks=stop(PAT_FT), verbose=0)
        pv3 = ft.predict(F2tr[iva].astype("float32"), verbose=0); pt3 = ft.predict(F2te.astype("float32"), verbose=0)
        cfg = f"T-{tag}"
        rows.append(dict(config=cfg, fold=fold, val_acc=accuracy_score(ytr[iva], pv3.argmax(1)),
                         val_loss=min(hf.history["val_loss"]), epochs=len(hf.history["loss"]),
                         **{"test_" + k: v for k, v in metrics(yte, pt3).items()}))
        store[(cfg, fold)] = dict(proba=pt3, hist=hf.history)
        log(f"fold {fold} {tag}:", {r["config"]: round(r["test_acc"], 3) for r in rows[-4 if aug else -3:]})
        keras.backend.clear_session()
        pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "hasil_per_fold.csv"), index=False)


[   128s] fold 1 noaug: {'F-noaug': 0.938, 'T-noaug': 0.906}
[   395s] fold 1 aug: {'T-noaug': 0.906, 'F-aug': 0.922, 'F-aug-/255': 0.484, 'T-aug': 0.938}
[   487s] fold 2 noaug: {'T-aug': 0.938, 'F-noaug': 0.922, 'T-noaug': 0.953}
[   810s] fold 2 aug: {'T-noaug': 0.953, 'F-aug': 0.938, 'F-aug-/255': 0.422, 'T-aug': 0.953}
[   874s] fold 3 noaug: {'T-aug': 0.953, 'F-noaug': 0.922, 'T-noaug': 0.922}
[  1179s] fold 3 aug: {'T-noaug': 0.922, 'F-aug': 0.953, 'F-aug-/255': 0.438, 'T-aug': 0.938}
[  1257s] fold 4 noaug: {'T-aug': 0.938, 'F-noaug': 0.938, 'T-noaug': 0.938}
[  1460s] fold 4 aug: {'T-noaug': 0.938, 'F-aug': 0.938, 'F-aug-/255': 0.5, 'T-aug': 0.969}
[  1525s] fold 5 noaug: {'T-aug': 0.969, 'F-noaug': 0.953, 'T-noaug': 0.953}
[  1845s] fold 5 aug: {'T-noaug': 0.953, 'F-aug': 0.938, 'F-aug-/255': 0.359, 'T-aug': 0.938}


## 9. Ringkasan, model terbaik (dipilih dari validasi), tabel, dan gambar

In [ ]:
# ------------------------------------------------------------------ 5. ringkasan
R = pd.DataFrame(rows)
order = ["F-aug-/255", "F-noaug", "F-aug", "T-noaug", "T-aug"]
def ms(s, pct=True):
    return f"{100*s.mean():.1f} ± {100*s.std(ddof=1):.1f}" if pct else f"{s.mean():.3f} ± {s.std(ddof=1):.3f}"
summ = pd.DataFrame([{"Konfigurasi": c, "Val acc (%)": ms(g.val_acc), "Test acc (%)": ms(g.test_acc),
                      "Test precision (w)": ms(g.test_prec_w, False), "Test F1 (w)": ms(g.test_f1_w, False),
                      "Test AUC": ms(g.test_auc, False), "Epoch": f"{g.epochs.mean():.1f}"}
                     for c in order for g in [R[R.config == c]]])
summ.to_csv(os.path.join(OUT_DIR, "ringkasan.csv"), index=False)
# model terbaik DIPILIH dari rerata akurasi validasi (bukan uji), fold terbaiknya dari akurasi validasi
cfgs = [c for c in order if c != "F-aug-/255"]
best_cfg = R[R.config.isin(cfgs)].groupby("config").val_acc.mean().idxmax()
bf = int(R[R.config == best_cfg].sort_values("val_acc").iloc[-1].fold)
proba = store[(best_cfg, bf)]["proba"]; pred = proba.argmax(1)
n_ok = int((pred == yte).sum()); lo, hi = wilson(n_ok, len(yte))
cm = confusion_matrix(yte, pred)
pc = precision_recall_fscore_support(yte, pred, zero_division=0)
pd.DataFrame({"Kelas": [CN.get(c, c) for c in classes], "Precision": pc[0], "Recall": pc[1], "F1": pc[2], "Support": pc[3]}
             ).to_csv(os.path.join(OUT_DIR, "per_kelas_terbaik.csv"), index=False)
# matriks kebingungan gabungan seluruh fold untuk konfigurasi terbaik
cm_all = sum(confusion_matrix(yte, store[(best_cfg, f)]["proba"].argmax(1)) for f in range(1, N_FOLDS + 1))
json.dump(dict(best_cfg=best_cfg, best_fold=bf, test_acc=n_ok / len(yte), n_correct=n_ok, wilson=[lo, hi],
               cm=cm.tolist(), cm_pooled=cm_all.tolist(), classes=classes), open(os.path.join(OUT_DIR, "terbaik.json"), "w"))
# baseline isyarat tingkat rendah: regresi logistik pada rerata RGB (uji kebocoran/pengganggu akuisisi)
mt = lambda X: X.reshape(len(X), -1, 3).mean(1) / 255.0
lr = LogisticRegression(max_iter=1000).fit(mt(Xtr), ytr)
json.dump(dict(rgb_only_test_acc=float(lr.score(mt(Xte), yte))), open(os.path.join(OUT_DIR, "baseline_rgb.json"), "w"))
# gambar
fig, ax = plt.subplots(figsize=(7, 3.4))
for c, mk in zip(cfgs + ["F-aug-/255"], "osd^v"):
    g = R[R.config == c].sort_values("fold")
    ax.plot(g.fold, 100 * g.test_acc, marker=mk, label=c)
ax.set_xlabel("Fold"); ax.set_ylabel("Test accuracy (%)"); ax.set_xticks(range(1, N_FOLDS + 1)); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "fig_acc_per_fold.png"), dpi=200)
fig, ax = plt.subplots(figsize=(4.2, 3.8)); ax.imshow(cm, cmap="Blues")
lab = [CN.get(c, c) for c in classes]; ax.set_xticks(range(3)); ax.set_xticklabels(lab, fontsize=8); ax.set_yticks(range(3)); ax.set_yticklabels(lab, fontsize=8)
for i in range(3):
    for j in range(3): ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "fig_confusion.png"), dpi=200)
hs = store[(best_cfg, bf)]["hist"]
fig, ax = plt.subplots(1, 2, figsize=(7, 3)); ax[0].plot(hs["loss"], label="train"); ax[0].plot(hs["val_loss"], label="val"); ax[0].set_title("Loss", fontsize=9)
ax[1].plot(hs["accuracy"], label="train"); ax[1].plot(hs["val_accuracy"], label="val"); ax[1].set_title("Accuracy", fontsize=9)
for a in ax: a.set_xlabel("Epoch"); a.legend(fontsize=7); a.grid(alpha=.3)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "fig_curves.png"), dpi=200)
log("SELESAI | terbaik:", best_cfg, "fold", bf, "| uji", n_ok, "/", len(yte), f"| Wilson {lo:.1f}-{hi:.1f}")
print(summ.to_string(index=False))


[  1847s] SELESAI | terbaik: T-aug fold 5 | uji 60 / 64 | Wilson 85.0-97.5
Konfigurasi Val acc (%) Test acc (%) Test precision (w)   Test F1 (w)      Test AUC Epoch
 F-aug-/255  49.6 ± 8.4   44.1 ± 5.6      0.462 ± 0.146 0.364 ± 0.100 0.717 ± 0.027  23.2
    F-noaug  94.3 ± 1.7   93.4 ± 1.3      0.935 ± 0.013 0.934 ± 0.013 0.983 ± 0.004  64.2
      F-aug  94.3 ± 3.0   93.8 ± 1.1      0.939 ± 0.011 0.937 ± 0.011 0.987 ± 0.006  42.2
    T-noaug  94.7 ± 1.8   93.4 ± 2.0      0.936 ± 0.020 0.934 ± 0.020 0.984 ± 0.002   5.2
      T-aug  95.1 ± 2.3   94.7 ± 1.4      0.950 ± 0.015 0.947 ± 0.014 0.992 ± 0.005   6.2


## 10. Unduh hasil

In [ ]:
!zip -qr hasil_resnet50.zip hasil_resnet50 -x 'hasil_resnet50/split/*' 'hasil_resnet50/cache.npz'
from google.colab import files
files.download('hasil_resnet50.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>